# APIs and HTTP

## Learning objectives

By the end of this notebook you will be able to:

- make a GET request with `requests` and pass query parameters safely;
- parse a JSON response into Python dictionaries and lists;
- implement pagination with `_page` and `_limit` parameters;
- recognise the common request failures and handle them with `try` / `except`;
- write a helper that returns a useful value and never crashes the notebook when offline.

## Concept

Much of the world's data is served over HTTP as JSON rather than shipped as a file. This
notebook uses the `requests` library to call a public, no-key JSON API, page through a long
result set, and — crucially — fail gracefully when the network is unavailable. It is the only
notebook in this module that needs a connection, so it is written to degrade to a clear message
rather than a traceback.

An **API** (application programming interface) is a contract that lets one program ask another
for data. A **REST** API exposes resources at URLs and uses HTTP verbs: `GET` to read, `POST` to
create, and so on. A response carries a **status code** — `200` means success, `4xx` means the
request was wrong, `5xx` means the server failed — plus a body, often JSON.

`requests.get(url, params={...})` builds the query string for you, which is safer than gluing
`?a=b&c=d` together by hand. `response.json()` parses the body, and `response.raise_for_status()`
turns an error status into an exception so the failure cannot pass silently.

**Pagination** exists because APIs rarely return everything at once. A common scheme is
`_page` (which page) and `_limit` (how many items per page), which is exactly what the public
posts endpoint uses. You keep requesting pages until a page comes back empty.

Good network code assumes failure. A timeout bounds how long you wait, a `try` / `except` around
the call keeps the program alive, and the helper returns an empty list so the rest of the
notebook can keep running.

## Worked example

### Imports and configuration

The import is guarded so the notebook still explains itself if `requests` is not installed.

In [1]:
from collections import Counter

try:
    import requests
except ImportError:  # pragma: no cover - depends on the environment
    requests = None
    print("requests is not installed; run: pip install requests")

BASE_URL = "https://jsonplaceholder.typicode.com/posts"
TIMEOUT = 10

print("requests available:", requests is not None)

requests available: True


### A helper that never crashes

The function below is the pattern we reuse everywhere. It catches `requests.RequestException`,
which is the base class for every network error, and reports what happened.

In [2]:
def fetch_posts(page=1, limit=5, base_url=BASE_URL, timeout=TIMEOUT):
    """Return a list of posts, or an empty list if the request cannot be made."""
    if requests is None:
        print("cannot fetch: the requests library is missing")
        return []
    try:
        response = requests.get(
            base_url,
            params={"_page": page, "_limit": limit},
            timeout=timeout,
        )
        response.raise_for_status()
    except requests.RequestException as exc:
        print(f"request failed ({type(exc).__name__}): {exc}")
        print("This notebook requires network access; later cells will report what was skipped.")
        return []
    return response.json()

### One page

`_page=1` and `_limit=5` ask for the first five posts. If we are offline the call returns `[]`
and the `ONLINE` flag records that so later cells can skip cleanly.

In [3]:
first_page = fetch_posts(page=1, limit=5)
ONLINE = len(first_page) > 0
print("posts returned:", len(first_page))
if ONLINE:
    for post in first_page:
        print(f"id={post['id']:>3}  user={post['userId']}  {post['title'][:55]}")

posts returned: 5
id=  1  user=1  sunt aut facere repellat provident occaecati excepturi 
id=  2  user=1  qui est esse
id=  3  user=1  ea molestias quasi exercitationem repellat qui ipsa sit
id=  4  user=1  eum et est occaecati
id=  5  user=1  nesciunt quas odio


The response is a list of dictionaries. The fields we use here are `id`, `userId`, and `title`;
the endpoint also returns a `body` field.

### Paginating a result set

There are 100 posts in total. Asking for three pages of five gives fifteen posts. We stop when a
page returns no items, which is more robust than hard-coding a total.

In [4]:
all_posts = []
if ONLINE:
    for page in range(1, 4):
        batch = fetch_posts(page=page, limit=5)
        if not batch:
            break
        all_posts.extend(batch)
        print(f"page {page}: +{len(batch)} posts (running total {len(all_posts)})")
else:
    print("offline: skipped pagination")

print("collected:", len(all_posts))

page 1: +5 posts (running total 5)


page 2: +5 posts (running total 10)


page 3: +5 posts (running total 15)
collected: 15


### A tiny analysis

Even a small payload supports a real question: which users wrote the posts we collected? A
`Counter` answers it without pandas.

In [5]:
if ONLINE and all_posts:
    per_user = Counter(post["userId"] for post in all_posts)
    for user, count in sorted(per_user.items()):
        print(f"user {user}: {count} posts")
else:
    print("offline: no posts to count")

user 1: 10 posts
user 2: 5 posts


### Handling a bad status code

A request can succeed at the network level and still be an error. `raise_for_status` converts a
`404` into an `HTTPError`, which the same `try` / `except` catches. The block is skipped
offline, since the error we want comes from the server, not from our code.

In [6]:
def fetch_one(post_id, base_url=BASE_URL, timeout=TIMEOUT):
    """Return a single post dictionary, or None on any failure."""
    if requests is None:
        return None
    try:
        response = requests.get(f"{base_url}/{post_id}", timeout=timeout)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as exc:
        print(f"could not fetch post {post_id}: {type(exc).__name__}")
        return None

if ONLINE:
    good = fetch_one(1)
    print("post 1 exists:", good is not None)
    missing = fetch_one(999_999)
    print("post 999999 result:", missing)
else:
    print("offline: skipped the status-code demonstration")

post 1 exists: True


could not fetch post 999999: HTTPError
post 999999 result: None


## Exercises

1. **Another page.** Fetch page 2 with `_limit=5` and print the titles. Confirm the ids do not
   overlap with page 1.
2. **Robust helper.** Write `fetch_page(page, limit, retries=3)` that retries a failed request up
   to `retries` times and returns `[]` once the attempts are exhausted.
3. **Count per user.** Collect the first ten posts with `_limit=10` and count how many each user
   wrote. Report the busiest user.

## Limitations

The posts endpoint is a mock service: it accepts writes but does not persist them, and it needs
no authentication. Real APIs add keys, OAuth tokens, rate limits, and cursor-based pagination
that uses values inside the response rather than page numbers. Requests can hang, so a timeout is
mandatory; a single retry is often sensible but hammering a failing server is not. JSON responses
can be deeply nested and inconsistent between versions, so production code validates the shape of
the payload before trusting it. This notebook treats every failure as "offline", which is fine
for teaching but would need finer reporting in a scheduled job.